## LSTM Language Model

instead of attending over the full context window, the LSTM compresses everything it's seen so far into a pair of vectors `(h, c)` — the hidden state and the cell state. These get updated one token at a time.

In [4]:
import torch
import torch.nn as nn
from torch.nn import functional as F

print(torch.__version__)
print('using', 'cuda' if torch.cuda.is_available() else 'cpu')

2.11.0+cu128
using cuda


In [5]:
# hyperparameters
batch_size    = 64
block_size    = 256     # BPTT window / training seq length (not an architectural limit like GPT)
max_iters     = 5000
eval_interval = 500
learning_rate = 1e-3
device        = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters    = 200
n_embd        = 256     # character embedding size
hidden_size   = 512     # LSTM hidden state size
n_layers      = 2       # stacked LSTM layers
dropout       = 0.3
# ------------

torch.manual_seed(1337)

In [6]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('total characters:', len(text))
print(text[:200])

total characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [7]:
# character-level tokenizer — exact same as gpt.py
chars      = sorted(list(set(text)))
vocab_size = len(chars)
print('vocab size:', vocab_size)
print(''.join(chars))

stoi   = {ch: i for i, ch in enumerate(chars)}
itos   = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

# sanity check
print(encode('hello'))
print(decode(encode('hello')))

vocab size: 65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[46, 43, 50, 50, 53]
hello


In [8]:
# train / val split
data = torch.tensor(encode(text), dtype=torch.long)
n    = int(0.9 * len(data))   # 90% train, 10% val
train_data = data[:n]
val_data   = data[n:]

print('train tokens:', len(train_data))
print('val tokens:', len(val_data))

train tokens: 1003854
val tokens: 111540


In [9]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix   = torch.randint(len(data) - block_size, (batch_size,))
    x    = torch.stack([data[i : i+block_size]     for i in ix])
    y    = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch('train')
print('x shape:', xb.shape)
print('y shape:', yb.shape)

x shape: torch.Size([64, 256])
y shape: torch.Size([64, 256])


In [10]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y      = get_batch(split)
            _, loss   = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

### The model

The whole Transformer from gpt.py (Head, MultiHeadAttention, FeedForward, Block, position embeddings) is replaced by a single `nn.LSTM` call.

Two things to know about PyTorch's LSTM:
- `batch_first=True` → input/output tensors are `(B, T, C)` not `(T, B, C)`
- the hidden state `(h, c)` has shape `(n_layers, B, hidden_size)` — note: NOT affected by `batch_first`

In [11]:
class LSTMLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.lstm    = nn.LSTM(n_embd, hidden_size, n_layers,
                               dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.lm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        x = self.token_embedding(idx)          # (B, T, n_embd)
        x = self.dropout(x)

        # reset hidden state at the start of each new batch
        h0 = torch.zeros(n_layers, B, hidden_size, device=idx.device)
        c0 = torch.zeros(n_layers, B, hidden_size, device=idx.device)

        x, _  = self.lstm(x, (h0, c0))        # x: (B, T, hidden_size)
        x      = self.dropout(x)
        logits = self.lm_head(x)               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        """
        unlike GPT's generate() which crops the context and re-runs attention
        every step, here we carry (h, c) forward between steps.
        one token in → one token out → update hidden state → repeat.
        """
        B = idx.shape[0]
        h = torch.zeros(n_layers, B, hidden_size, device=idx.device)
        c = torch.zeros(n_layers, B, hidden_size, device=idx.device)

        # prime (h, c) on the seed context (all tokens except the last)
        if idx.shape[1] > 1:
            seed_emb  = self.token_embedding(idx[:, :-1])   # (B, seed_len-1, n_embd)
            _, (h, c) = self.lstm(seed_emb, (h, c))

        for _ in range(max_new_tokens):
            x          = self.token_embedding(idx[:, -1:])   # (B, 1, n_embd)
            out, (h,c) = self.lstm(x, (h, c))                # out: (B, 1, hidden_size)
            logits     = self.lm_head(out[:, -1, :])          # (B, vocab_size)
            probs      = F.softmax(logits, dim=-1)
            idx_next   = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx        = torch.cat([idx, idx_next], dim=1)

        return idx

In [12]:
model = LSTMLanguageModel().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M parameters')
print(model)

3.728193 M parameters
LSTMLanguageModel(
  (token_embedding): Embedding(65, 256)
  (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
  (dropout): Dropout(p=0.3, inplace=False)
  (lm_head): Linear(in_features=512, out_features=65, bias=True)
)


In [13]:
# quick sanity check before training
xb, yb     = get_batch('train')
logits, loss = model(xb, yb)
print('logits shape:', logits.shape)
print('loss before training:', loss.item())

logits shape: torch.Size([64, 256, 65])
loss before training: 4.175232887268066


In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

for i in range(max_iters):
    if i % eval_interval == 0 or i == max_iters - 1:
        losses = estimate_loss()
        print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    # clip gradients — LSTMs can have exploding gradients without this
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

step 0: train loss 4.1750, val loss 4.1746
step 500: train loss 1.3848, val loss 1.5673
step 1000: train loss 1.2446, val loss 1.4711
step 1500: train loss 1.1699, val loss 1.4415
step 2000: train loss 1.1113, val loss 1.4326
step 2500: train loss 1.0617, val loss 1.4329
step 3000: train loss 1.0122, val loss 1.4402
step 3500: train loss 0.9734, val loss 1.4599
step 4000: train loss 0.9377, val loss 1.4732
step 4500: train loss 0.9061, val loss 1.4892
step 4999: train loss 0.8755, val loss 1.5068


In [15]:
# generate some Shakespeare
# model.eval() disables LSTM inter-layer dropout so this matches
# the clean model that estimate_loss() was measuring
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


Have I not dined to bear? What's thy name?

LADY CAPULET:
Say, good night.
What say'st thou not? Richard, dost thou say 'tis one?
Within this friar fill's heaven from the open sigh,
So it extremes all at once, or else in a
revenges sure.

MENENIUS:
This danger which
it is: or is no man promised?

VOLUMNIA:
Ay, sir; a while: if you'll bestolet them,
But yet you might pray of your patience speak;
For that is here worse from these amperied:
How now, virgin, abides, the hardel number.
As if you lay 


In [16]:
# optionally seed with some actual text
model.eval()
seed_text  = 'ROMEO:'
seed_idx   = torch.tensor([encode(seed_text)], dtype=torch.long, device=device)  # (1, 6)
generated  = model.generate(seed_idx, max_new_tokens=300)[0].tolist()
print(decode(generated))

ROMEO:
I thought you do well between it. At a fit
false air, to murder him on me, provost, I would
by him, for one come 'not.'

DUKE VINCENTIO:
No.

PETER:
Do you seem in inkent
That been appear'd?

First Lord:
Here is come but thy windows; but indeed
I'll speak to them, my lord.

VALERIA:
On, pity! not i
